# 2. The model

This notebook builds the network from the final settings, follows a batch through it, and trains two seeds on the synthetic package (a few seconds on a CPU). It uses no real data and shows no results.

In [1]:
import tempfile
from pathlib import Path
import numpy as np
import torch
from deep_multimodal.settings import load_settings
from deep_multimodal.model import DeepMultimodalModel, ModelSettings
from deep_multimodal.synthetic import write_synthetic_package
from deep_multimodal.data import load_fold
from deep_multimodal.training import TrainingSettings, train_two_stage, predict_logits

model_settings, training, config = load_settings()
model_settings, training

(ModelSettings(width=64, layers=2, dropout=0.1, heads=4, modality_dropout=0.1, auxiliary_weight=0.2, use_forecast=True),
 TrainingSettings(learning_rate=0.0003, weight_decay=0.01, max_epochs=20, patience=2, minimum_gain=1e-06, batch_size=4096, seeds=(0, 1, 2, 3, 4, 5, 6, 7)))

## Size

The final settings (width 64, 2 layers, 4 heads) give 296,587 parameters.

In [2]:
model = DeepMultimodalModel(model_settings)
print(sum(p.numel() for p in model.parameters()), 'parameters')
for name, module in model.named_children():
    print(f'{name:32s}', sum(p.numel() for p in module.parameters()) if hasattr(module, 'parameters') else '')

296587 parameters
fire_channel_projection          320
lag_embedding                    960
position_embedding               3136
fire_blocks                      66944
weather_gru                      44736
weather_segment_embedding        768
forecast_conv                    13376
forecast_daily_projection        1344
static_blocks                    66944
stream_type_embedding            256
query_projection                 4480
fusion_blocks                    84352
output_head                      4353
skip_linear                      7
auxiliary_head                   130


## One batch

Four streams become tokens: fire history (up to 33), weather (12), forecast (6) and static (23). A query token attends over all of them in two cross-attention layers.

In [3]:
folder = Path(tempfile.mkdtemp()) / 'package'
write_synthetic_package(folder)
fold = load_fold(folder, 2022, torch.device('cpu'))
batch = fold.validation.batch(torch.arange(8))
model.eval()
with torch.no_grad():
    query, condition, parts = model._streams(batch, training=False)
print('query', tuple(query.shape), '| conditioning', tuple(condition.shape))
for (tokens, padding), name in zip(parts, ['fire history', 'weather', 'forecast', 'static']):
    print(f'{name:13s} tokens', tuple(tokens.shape), '| masked entries:', int(padding.sum()))

query (8, 1, 64) | conditioning (8, 66)
fire history  tokens (8, 33, 64) | masked entries: 149
weather       tokens (8, 12, 64) | masked entries: 0
forecast      tokens (8, 6, 64) | masked entries: 0
static        tokens (8, 23, 64) | masked entries: 0


The output is one logit per row; the probability is its sigmoid. The auxiliary head (detection size on the target day) is used in training only.

In [4]:
with torch.no_grad():
    logit, auxiliary = model(batch)
print('probabilities of the first rows (untrained model):', torch.sigmoid(logit)[:4].numpy().round(3))

probabilities of the first rows (untrained model): [0.485 0.471 0.506 0.473]


## Training two stages

Stage 1 stops on the last fit year; stage 2 retrains on all fit rows for the chosen number of epochs. Here a tiny network is trained on random data to show the mechanics.

In [5]:
small = ModelSettings(width=16, layers=1, heads=2, dropout=0.0)
fast = TrainingSettings(max_epochs=3, batch_size=256, seeds=(0, 1))
trained, epochs = train_two_stage(fold, small, fast, 0, torch.device('cpu'))
print('epochs chosen in stage 1:', epochs)

epochs chosen in stage 1: 3


## An ensemble

The final model averages the probabilities of 8 seeds. Two seeds are averaged here.

In [6]:
probabilities = []
for seed in fast.seeds:
    network, _ = train_two_stage(fold, small, fast, seed, torch.device('cpu'))
    probabilities.append(1 / (1 + np.exp(-predict_logits(network, fold.validation))))
ensemble = np.mean(probabilities, 0)
print('ensemble of', len(probabilities), 'seeds; first probabilities:', ensemble[:4].round(3))

ensemble of 2 seeds; first probabilities: [0.447 0.444 0.442 0.459]
